## START

In [9]:
# Loading the documents
# We'll use helper files from module 01 and this module.

# If you don't have them in your notebook directory, download them:

# PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'

# !wget {PREFIX}/01-agentic-rag/code/ingest.py
# !wget {PREFIX}/01-agentic-rag/code/rag_helper.py
# !wget {PREFIX}/04-evaluation/code/evaluation_utils.py
# Then load the FAQ data:

from ingest import load_faq_data
documents = load_faq_data()
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [10]:
# We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ dataset contains documents from multiple courses. 
# Generating five questions for every document would take longer and cost more.

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [11]:
# We'll use these documents from now on so let's name them as documents

documents = documents_llm
# Each document already has an id field:

doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Generating questions with structured output

In [13]:
# Generating questions with structured output
# We use an LLM to generate questions for each document.

# This is the first time we're using structured output in the course. With structured output, we ask the LLM to return data 
# in a specific format instead of free-form text. For example, instead of getting a paragraph that contains questions, 
# we can ask for a Python object with a questions field.

# This is useful when code will process the output. The model returns the same structure every time. We can access the 
# generated questions directly instead of parsing text manually.

# We want the output as a list of strings, so we define that structure with a Pydantic model:

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# The instructions for the LLM:

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

# We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users 
# won't phrase their questions the same way as the FAQ.

# Call the LLM for one document:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Prepare the document as JSON:

import json

user_prompt = json.dumps(doc)
# Create the messages:

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

# The parsed object is available in response.output_parsed:

result = response.output_parsed

print(result)
# We can access the list directly:

print(result.questions)
# You should see 5 questions that relate to the first FAQ document.

questions=['I found this course late — can I still join it, and is there anything I need to do if I want a certificate?', 'Is it too late to start the course now, or can I still enroll even after it started?', 'If I join the course late, do I still qualify for the certificate as long as I submit the project on time?', 'Can I start the course after it has begun, and what are the rules for getting certified?', 'I just heard about the course — is it still open for new students, and when do I need to turn in the project for the certificate?']
['I found this course late — can I still join it, and is there anything I need to do if I want a certificate?', 'Is it too late to start the course now, or can I still enroll even after it started?', 'If I join the course late, do I still qualify for the certificate as long as I submit the project on time?', 'Can I start the course after it has begun, and what are the rules for getting certified?', 'I just heard about the course — is it still open for